# C2.9 · Research as institutional capital

**Function C — Offensive Security & Research → The Security Researcher**  ·  *Both directions*

Builds on **[C2.8 · From finding to control](https://spbreed.github.io/cyber-commons/lessons/C2.8.html)**.

| | |
|---|---|
| Open-source tooling | git |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The test of a research programme is not what it discovered. It is what still
protects you after the person who discovered it has left.

Findings land in artefacts of very different durability, and the difference is
stark:

| Landed as | Survives staff turnover? | Survives a refactor? |
|---|---|---|
| a chat thread | no | no |
| a slide deck | technically | no |
| a repro card | yes | no |
| a regression case in CI | yes | **yes — it fails the build** |
| a control + its eval case | yes | yes |

Only the last two are institutional capital. Everything above them is a record
of work, not protection.

This is also the lesson where the programme gets measured, because "how much
research did we do" is the wrong question and "how much of it is still holding"
is the right one.

## 2 · Demo — score the artefact ladder

In [ ]:
LADDER = {
 "chat thread":            (0, "gone at the next retention sweep"),
 "slide deck":             (1, "survives; nobody re-runs it"),
 "written repro card":     (2, "someone else can reproduce it"),
 "detection rule":         (3, "fires if the precondition recurs"),
 "regression case in CI":  (4, "fails the build when the finding returns"),
 "control + eval case":    (5, "prevents it AND proves it stays prevented"),
}
print(f"{'artefact':26s}{'durability':>11}  what it buys")
print("-" * 78)
for k, (score, buys) in LADDER.items():
    print(f"{k:26s}{score:>11}  {buys}")

## 3 · Where it breaks — a productive year that protects nothing

In [ ]:
YEAR = [
 ("diff-borne approval",        "control + eval case"),
 ("token widening at hop 3",    "control + eval case"),
 ("metadata reachable in staging","regression case in CI"),
 ("prompt leak via error text", "detection rule"),
 ("model drift after upgrade",  "slide deck"),
 ("odd retry storm",            "chat thread"),
 ("MCP package with no signature","written repro card"),
 ("agent scored as human",      "chat thread"),
 ("eval corpus imbalance",      "slide deck"),
 ("SSRF via allowlisted host",  "written repro card"),
]
total = sum(LADDER[a][0] for _, a in YEAR)
maxi  = 5 * len(YEAR)
print(f"{'finding':34s}{'landed as':26s}{'score':>6}")
print("-" * 68)
for name, artefact in YEAR:
    print(f"{name:34s}{artefact:26s}{LADDER[artefact][0]:>6}")
print(f"\nfindings: {len(YEAR)}   durability {total}/{maxi} = {total/maxi:.0%}")

survives_turnover = sum(1 for _, a in YEAR if LADDER[a][0] >= 2)
survives_refactor = sum(1 for _, a in YEAR if LADDER[a][0] >= 4)
print(f"survives staff turnover: {survives_turnover}/{len(YEAR)}")
print(f"survives a refactor:     {survives_refactor}/{len(YEAR)}")
print("\nTen findings. Three of them will still protect you in two years.")

## 4 · The control — a promotion rule, applied at closure

In [ ]:
def promotion_target(severity, recurring, cheap_to_test):
    """Where a finding MUST land before it may be closed."""
    if severity in ("critical", "high"):
        return "control + eval case"
    if recurring:
        return "regression case in CI"
    if cheap_to_test:
        return "regression case in CI"
    return "written repro card"

CANDIDATES = [
 ("diff-borne approval",         "critical", True,  True),
 ("model drift after upgrade",   "medium",   True,  True),
 ("odd retry storm",             "low",      False, False),
 ("SSRF via allowlisted host",   "high",     False, True),
]
print(f"{'finding':32s}{'severity':10s}{'must land as':26s}")
print("-" * 72)
for name, sev, recurring, cheap in CANDIDATES:
    print(f"{name:32s}{sev:10s}{promotion_target(sev, recurring, cheap):26s}")

def may_close(landed_as, required):
    return LADDER[landed_as][0] >= LADDER[required][0]

print("\nclosure check against what actually happened:")
ACTUAL = {"diff-borne approval": "control + eval case",
          "model drift after upgrade": "slide deck",
          "odd retry storm": "chat thread",
          "SSRF via allowlisted host": "written repro card"}
for name, sev, rec, cheap in CANDIDATES:
    req = promotion_target(sev, rec, cheap)
    ok = may_close(ACTUAL[name], req)
    print(f"   {name:32s} landed={ACTUAL[name]:24s} {'CLOSE' if ok else 'REOPEN'}")
reopened = [n for n, s, r, c in CANDIDATES
            if not may_close(ACTUAL[n], promotion_target(s, r, c))]
print(f"\nmust reopen: {reopened}")
assert "model drift after upgrade" in reopened

## What you just proved

The ladder scores six artefact types 0–5. The year's ten findings score 26/50 durability, with 6 surviving staff turnover and only 3 surviving a refactor. The promotion rule requires critical findings to land as a control plus eval case, and the closure check reopens the drift and retry-storm findings.

## Your turn

Apply the promotion rule to your open findings backlog. The ones that were closed below their required artefact are the ones you will rediscover — budget for finding them twice, or promote them now.

---

**Next → [D1.1 · From alert queue to loop operator](https://spbreed.github.io/cyber-commons/lessons/D1.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*